In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pathlib import Path
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

spark = SparkSession.builder.master("local[*]").appName("AzureRetailLakehouse").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/03 15:09:45 WARN Utils: Your hostname, Branimirs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.199 instead (on interface en0)
26/08/03 15:09:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/03 15:09:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applic

In [2]:
orders_data = [
    (1001, 1, "2026-07-01", "completed", 120.50),
    (1002, 2, "2026-07-02", "completed", 75.00),
    (1003, 1, "2026-07-03", "cancelled", 50.00),
    (1004, 3, "2026-07-04", "completed", None),
    (1005, 4, "invalid-date", "pending", 210.25),
    (1006, 2, "2026-07-06", "completed", 90.00),
    (1007, None, "2026-07-07", "completed", 45.50),
    (1008, 5, "2026-07-08", "shipped", 180.00),
    (1008, 5, "2026-07-08", "shipped", 180.00),
    (1009, 3, "2026-07-09", "completed", 125.00),
]

columns = [
    "order_id",
    "customer_id",
    "order_date",
    "status",
    "order_amount",
]

orders_df = spark.createDataFrame(orders_data, columns)
orders_df.show()

+--------+-----------+------------+---------+------------+
|order_id|customer_id|  order_date|   status|order_amount|
+--------+-----------+------------+---------+------------+
|    1001|          1|  2026-07-01|completed|       120.5|
|    1002|          2|  2026-07-02|completed|        75.0|
|    1003|          1|  2026-07-03|cancelled|        50.0|
|    1004|          3|  2026-07-04|completed|        NULL|
|    1005|          4|invalid-date|  pending|      210.25|
|    1006|          2|  2026-07-06|completed|        90.0|
|    1007|       NULL|  2026-07-07|completed|        45.5|
|    1008|          5|  2026-07-08|  shipped|       180.0|
|    1008|          5|  2026-07-08|  shipped|       180.0|
|    1009|          3|  2026-07-09|completed|       125.0|
+--------+-----------+------------+---------+------------+



In [3]:
orders_df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- order_amount: double (nullable = true)



In [4]:
orders_df.count()

10

In [5]:
orders_df.select("order_id", "customer_id", "order_amount").show()

+--------+-----------+------------+
|order_id|customer_id|order_amount|
+--------+-----------+------------+
|    1001|          1|       120.5|
|    1002|          2|        75.0|
|    1003|          1|        50.0|
|    1004|          3|        NULL|
|    1005|          4|      210.25|
|    1006|          2|        90.0|
|    1007|       NULL|        45.5|
|    1008|          5|       180.0|
|    1008|          5|       180.0|
|    1009|          3|       125.0|
+--------+-----------+------------+



In [6]:
orders_df.select(
    "order_id",
    F.col("order_amount"),
    (F.col("order_amount") * 1.20).alias("amount_with_tax"),
).show()

+--------+------------+------------------+
|order_id|order_amount|   amount_with_tax|
+--------+------------+------------------+
|    1001|       120.5|             144.6|
|    1002|        75.0|              90.0|
|    1003|        50.0|              60.0|
|    1004|        NULL|              NULL|
|    1005|      210.25|252.29999999999998|
|    1006|        90.0|             108.0|
|    1007|        45.5|              54.6|
|    1008|       180.0|             216.0|
|    1008|       180.0|             216.0|
|    1009|       125.0|             150.0|
+--------+------------+------------------+



In [7]:
completed_orders = orders_df.filter(
    F.col("status") == "completed"
).show()




+--------+-----------+----------+---------+------------+
|order_id|customer_id|order_date|   status|order_amount|
+--------+-----------+----------+---------+------------+
|    1001|          1|2026-07-01|completed|       120.5|
|    1002|          2|2026-07-02|completed|        75.0|
|    1004|          3|2026-07-04|completed|        NULL|
|    1006|          2|2026-07-06|completed|        90.0|
|    1007|       NULL|2026-07-07|completed|        45.5|
|    1009|          3|2026-07-09|completed|       125.0|
+--------+-----------+----------+---------+------------+



In [8]:
orders_with_tax = orders_df.withColumn("amount_with_tax", F.col("order_amount") * 1.20)
orders_with_tax.show()

+--------+-----------+------------+---------+------------+------------------+
|order_id|customer_id|  order_date|   status|order_amount|   amount_with_tax|
+--------+-----------+------------+---------+------------+------------------+
|    1001|          1|  2026-07-01|completed|       120.5|             144.6|
|    1002|          2|  2026-07-02|completed|        75.0|              90.0|
|    1003|          1|  2026-07-03|cancelled|        50.0|              60.0|
|    1004|          3|  2026-07-04|completed|        NULL|              NULL|
|    1005|          4|invalid-date|  pending|      210.25|252.29999999999998|
|    1006|          2|  2026-07-06|completed|        90.0|             108.0|
|    1007|       NULL|  2026-07-07|completed|        45.5|              54.6|
|    1008|          5|  2026-07-08|  shipped|       180.0|             216.0|
|    1008|          5|  2026-07-08|  shipped|       180.0|             216.0|
|    1009|          3|  2026-07-09|completed|       125.0|      

In [9]:
clean_orders = orders_df.filter(F.col("customer_id").isNotNull()).withColumn("order_date", F.expr("try_cast(order_date as date)")).dropDuplicates(["order_id"])

clean_orders.show()

+--------+-----------+----------+---------+------------+
|order_id|customer_id|order_date|   status|order_amount|
+--------+-----------+----------+---------+------------+
|    1001|          1|2026-07-01|completed|       120.5|
|    1002|          2|2026-07-02|completed|        75.0|
|    1003|          1|2026-07-03|cancelled|        50.0|
|    1004|          3|2026-07-04|completed|        NULL|
|    1005|          4|      NULL|  pending|      210.25|
|    1006|          2|2026-07-06|completed|        90.0|
|    1008|          5|2026-07-08|  shipped|       180.0|
|    1009|          3|2026-07-09|completed|       125.0|
+--------+-----------+----------+---------+------------+



In [10]:
parsed_orders = orders_df.withColumn("parsed_order_date", F.expr("try_cast(order_date as date)"))
parsed_orders.show()

+--------+-----------+------------+---------+------------+-----------------+
|order_id|customer_id|  order_date|   status|order_amount|parsed_order_date|
+--------+-----------+------------+---------+------------+-----------------+
|    1001|          1|  2026-07-01|completed|       120.5|       2026-07-01|
|    1002|          2|  2026-07-02|completed|        75.0|       2026-07-02|
|    1003|          1|  2026-07-03|cancelled|        50.0|       2026-07-03|
|    1004|          3|  2026-07-04|completed|        NULL|       2026-07-04|
|    1005|          4|invalid-date|  pending|      210.25|             NULL|
|    1006|          2|  2026-07-06|completed|        90.0|       2026-07-06|
|    1007|       NULL|  2026-07-07|completed|        45.5|       2026-07-07|
|    1008|          5|  2026-07-08|  shipped|       180.0|       2026-07-08|
|    1008|          5|  2026-07-08|  shipped|       180.0|       2026-07-08|
|    1009|          3|  2026-07-09|completed|       125.0|       2026-07-09|

In [11]:
parsed_orders.select("order_id", "order_date", "parsed_order_date").show()

+--------+------------+-----------------+
|order_id|  order_date|parsed_order_date|
+--------+------------+-----------------+
|    1001|  2026-07-01|       2026-07-01|
|    1002|  2026-07-02|       2026-07-02|
|    1003|  2026-07-03|       2026-07-03|
|    1004|  2026-07-04|       2026-07-04|
|    1005|invalid-date|             NULL|
|    1006|  2026-07-06|       2026-07-06|
|    1007|  2026-07-07|       2026-07-07|
|    1008|  2026-07-08|       2026-07-08|
|    1008|  2026-07-08|       2026-07-08|
|    1009|  2026-07-09|       2026-07-09|
+--------+------------+-----------------+



In [12]:
valid_orders = (
    parsed_orders
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("parsed_order_date").isNotNull())
    .dropDuplicates(["order_id"])
    .drop("order_date")
    .withColumnRenamed("parsed_order_date", "order_date")
)

valid_orders.show()

+--------+-----------+---------+------------+----------+
|order_id|customer_id|   status|order_amount|order_date|
+--------+-----------+---------+------------+----------+
|    1001|          1|completed|       120.5|2026-07-01|
|    1002|          2|completed|        75.0|2026-07-02|
|    1003|          1|cancelled|        50.0|2026-07-03|
|    1004|          3|completed|        NULL|2026-07-04|
|    1006|          2|completed|        90.0|2026-07-06|
|    1008|          5|  shipped|       180.0|2026-07-08|
|    1009|          3|completed|       125.0|2026-07-09|
+--------+-----------+---------+------------+----------+



In [13]:
valid_orders.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- status: string (nullable = true)
 |-- order_amount: double (nullable = true)
 |-- order_date: date (nullable = true)



In [14]:
rejected_orders = (
    parsed_orders
    .filter(F.col("customer_id").isNotNull() | F.col("parsed_order_date").isNotNull())
    .withColumn(
        "rejected_reason",
        F.when(
            F.col("customer_id").isNull(),
            F.lit("Missing customer_id")
        )
        .when(
            F.col("parsed_order_date").isNull(),
            F.lit("Invalid order_date")
        )
        .otherwise(F.lit("Unknown error"))
    )
)

rejected_orders.show()

+--------+-----------+------------+---------+------------+-----------------+-------------------+
|order_id|customer_id|  order_date|   status|order_amount|parsed_order_date|    rejected_reason|
+--------+-----------+------------+---------+------------+-----------------+-------------------+
|    1001|          1|  2026-07-01|completed|       120.5|       2026-07-01|      Unknown error|
|    1002|          2|  2026-07-02|completed|        75.0|       2026-07-02|      Unknown error|
|    1003|          1|  2026-07-03|cancelled|        50.0|       2026-07-03|      Unknown error|
|    1004|          3|  2026-07-04|completed|        NULL|       2026-07-04|      Unknown error|
|    1005|          4|invalid-date|  pending|      210.25|             NULL| Invalid order_date|
|    1006|          2|  2026-07-06|completed|        90.0|       2026-07-06|      Unknown error|
|    1007|       NULL|  2026-07-07|completed|        45.5|       2026-07-07|Missing customer_id|
|    1008|          5|  2026-0

In [15]:
orders_path = Path("../data/raw/olist/olist_orders_dataset.csv").resolve()
print(orders_path)

/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/data/raw/olist/olist_orders_dataset.csv


In [16]:
orders_df = spark.read.option("header", True).option("inferSchema", False).csv(str(orders_path))

In [17]:
orders_df.show(5, truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|delivered   |2017-10-02 10:56:33     |2017-10-02 11:07:15|2017-10-04 19:55:00         |2017-10-10 21:25:13          |2017-10-18 00:00:00          |
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|delivered   |2018-07-24 20:41:37     |2018-07-26 03:24:27|2018-07-26 14:3

In [18]:
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)



In [19]:
print(f"Rows: {orders_df.count()}")
print(f"Columns: {len(orders_df.columns)}")
print(f"Column names: {orders_df.columns}")

Rows: 99441
Columns: 8
Column names: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


In [20]:
rows_count = orders_df.count()

distinct_orders_count = orders_df.select("order_id").distinct().count()

print("Rows count: ", rows_count)
print("Distinct order IDs: ", distinct_orders_count)
print("order_id is unique: ", distinct_orders_count == rows_count)

Rows count:  99441
Distinct order IDs:  99441
order_id is unique:  True


In [21]:
orders_status_counts = orders_df.groupBy('order_status').count().orderBy(F.col("count"), ascending=False)

print(orders_status_counts.show())

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+

None


In [22]:
delivered_without_delivery_dates = (
    orders_df
    .filter(F.col("order_status") == "delivered")
    .filter(F.col("order_delivered_customer_date").isNull())
)

delivered_without_delivery_dates.show(truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|2d1e2d5bf4dc7227b3bfebb81328c15f|ec05a6d8558c6455f0cbbd8a420ad34f|delivered   |2017-11-28 17:44:07     |2017-11-28 17:56:40|2017-11-30 18:12:23         |NULL                         |2017-12-18 00:00:00          |
|f5dd62b788049ad9fc0526e3ad11a097|5e89028e024b381dc84a13a3570decb4|delivered   |2018-06-20 06:58:43     |2018-06-20 07:19:05|2018-06-25 08:0

In [23]:
not_delivered_with_delivery_date = (
    orders_df
    .filter((F.col("order_status") != "delivered") & (F.col("order_delivered_customer_date").isNotNull()))
)

not_delivered_with_delivery_date.show(truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|1950d777989f6a877539f53795b4c3c3|1bccb206de9f0f25adc6871a1bcf77b2|canceled    |2018-02-19 19:48:52     |2018-02-19 20:56:05|2018-02-20 19:57:13         |2018-03-21 22:03:51          |2018-03-09 00:00:00          |
|dabf2b0e35b423f94618bf965fcb7514|5cdec0bb8cbdf53ffc8fdc212cd247c6|canceled    |2016-10-09 00:56:52     |2016-10-09 13:36:58|2016-10-13 13:3

In [24]:
print("Delivered without delivery date: ", delivered_without_delivery_dates.count())
print("Not delivered with delivery date: ", not_delivered_with_delivery_date.count())

Delivered without delivery date:  8
Not delivered with delivery date:  6


In [25]:
orders_with_quality = orders_df.withColumn(
    "delivery_status_quality",
    F.when(
        (F.col("order_status") == "delivered") & (F.col("order_delivered_customer_date").isNull()),
        F.lit("DELIVERED_WITHOUT_DATE")
    )
    .when(
        ((F.col("order_status") != "delivered") & (F.col("order_delivered_customer_date").isNotNull())),
        F.lit("DATE_WITHOUT_DELIVERED_STATUS")
    ).otherwise(F.lit("VALID"))

)

orders_with_quality.groupBy('delivery_status_quality').count().show(truncate=False)

+-----------------------------+-----+
|delivery_status_quality      |count|
+-----------------------------+-----+
|VALID                        |99427|
|DATE_WITHOUT_DELIVERED_STATUS|6    |
|DELIVERED_WITHOUT_DATE       |8    |
+-----------------------------+-----+



In [26]:
orders_df.columns

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

In [27]:
timestamp_columns = [
     'order_purchase_timestamp',
     'order_approved_at',
     'order_delivered_carrier_date',
     'order_delivered_customer_date',
     'order_estimated_delivery_date'
]

orders_typed = orders_with_quality

for col_name in timestamp_columns:
    orders_typed = orders_typed.withColumn(col_name, F.to_timestamp(F.col(col_name)))

orders_typed.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- delivery_status_quality: string (nullable = false)



In [28]:
orders_typed.select(
    'order_id',
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
).show(5, truncate=False)

+--------------------------------+------------------------+-----------------------------+-----------------------------+
|order_id                        |order_purchase_timestamp|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|2017-10-02 10:56:33     |2017-10-10 21:25:13          |2017-10-18 00:00:00          |
|53cdb2fc8bc7dce0b6741e2150273451|2018-07-24 20:41:37     |2018-08-07 15:27:45          |2018-08-13 00:00:00          |
|47770eb9100c2d0c44946d9cf07ec65d|2018-08-08 08:38:49     |2018-08-17 18:06:29          |2018-09-04 00:00:00          |
|949d5b44dbf5de918fe9c16f97b45f8a|2017-11-18 19:28:06     |2017-12-02 00:28:42          |2017-12-15 00:00:00          |
|ad21c59c0840e6cb83a9ceb5573f8159|2018-02-13 21:18:39     |2018-02-16 18:17:02          |2018-02-26 00:00:00          |
+--------------------------------+------

In [29]:
orders_metrics = (
    orders_typed
    .withColumn(
        "delivery_days",
        F.datediff(
            F.col("order_delivered_customer_date"),
            F.col("order_purchase_timestamp")
        )
    )
    .withColumn(
        "delivery_delays_days",
        F.datediff(
            F.col("order_delivered_customer_date"),
            F.col("order_estimated_delivery_date")
        )
    )
)

In [30]:
orders_metrics.select(
    'order_id',
    'order_status',
    'delivery_days',
    'delivery_delays_days',
).show(10, truncate=False)

+--------------------------------+------------+-------------+--------------------+
|order_id                        |order_status|delivery_days|delivery_delays_days|
+--------------------------------+------------+-------------+--------------------+
|e481f51cbdc54678b7cc49136f2d6af7|delivered   |8            |-8                  |
|53cdb2fc8bc7dce0b6741e2150273451|delivered   |14           |-6                  |
|47770eb9100c2d0c44946d9cf07ec65d|delivered   |9            |-18                 |
|949d5b44dbf5de918fe9c16f97b45f8a|delivered   |14           |-13                 |
|ad21c59c0840e6cb83a9ceb5573f8159|delivered   |3            |-10                 |
|a4591c265e18cb1dcee52889e2d8acc3|delivered   |17           |-6                  |
|136cce7faa42fdb2cefd53fdc79a6098|invoiced    |NULL         |NULL                |
|6514b8ad8028c9f2cc2374ded245783f|delivered   |10           |-12                 |
|76c6e866289321a7c93b82b54852dc33|delivered   |10           |-32                 |
|e69

In [31]:
delayed_orders = orders_metrics.filter(F.col('delivery_delays_days') > 0).count()
print(delayed_orders)

6535


In [32]:
delivered_orders = orders_metrics.filter(F.col("order_status") == "delivered")
late_delivered_orders = orders_metrics.filter(F.col('delivery_delays_days') > 0)

delivered_count = delivered_orders.count()
late_count = late_delivered_orders.count()

late_percentage = late_count / delivered_count * 100

print("Delivered orders: ", delivered_count)
print("Late delivered orders: ", late_count)
print(f"Late delivery percentage: {round(late_percentage, 2)} %")

Delivered orders:  96478
Late delivered orders:  6535
Late delivery percentage: 6.77 %


In [33]:
orders_metrics.filter(
    F.col("delivery_delays_days") > 0
).select(
    F.min("delivery_delays_days").alias("min_delay"),
    F.avg("delivery_delays_days").alias("avg_delay"),
    F.max("delivery_delays_days").alias("max_delay"),
).show()

+---------+----------------+---------+
|min_delay|       avg_delay|max_delay|
+---------+----------------+---------+
|        1|10.6203519510329|      188|
+---------+----------------+---------+



In [34]:
orders_metrics.columns

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'delivery_status_quality',
 'delivery_days',
 'delivery_delays_days']

In [35]:
orders_metrics.filter(
    F.col("delivery_delays_days") > 0
).select(
    'order_id',
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_delays_days"
).orderBy(
    F.col("delivery_delays_days").desc(),
).show(10, truncate=False)

+--------------------------------+------------------------+-----------------------------+-----------------------------+--------------------+
|order_id                        |order_purchase_timestamp|order_delivered_customer_date|order_estimated_delivery_date|delivery_delays_days|
+--------------------------------+------------------------+-----------------------------+-----------------------------+--------------------+
|1b3190b2dfa9d789e1f14c05b647a14a|2018-02-23 14:57:35     |2018-09-19 23:24:07          |2018-03-15 00:00:00          |188                 |
|ca07593549f1816d26a572e06dc1eab6|2017-02-21 23:31:27     |2017-09-19 14:36:39          |2017-03-22 00:00:00          |181                 |
|47b40429ed8cce3aee9199792275433f|2018-01-03 09:44:01     |2018-07-13 20:51:31          |2018-01-19 00:00:00          |175                 |
|2fe324febf907e3ea3f2aa9650869fa5|2017-03-13 20:17:10     |2017-09-19 17:00:07          |2017-04-05 00:00:00          |167                 |
|285ab9426d69

In [36]:
orders_schema = StructType([
    StructField("order_id", StringType(), nullable=False),
    StructField("customer_id", StringType(), nullable=False),
    StructField("order_status", StringType(), nullable=True),
    StructField("order_purchase_timestamp", TimestampType(), nullable=True),
    StructField("order_approved_at", TimestampType(), nullable=True),
    StructField("order_delivered_carrier_date", TimestampType(), nullable=True),
    StructField("order_delivered_customer_date", TimestampType(), nullable=True),
    StructField("order_estimated_delivery_date", TimestampType(), nullable=True),
])

In [37]:
orders_typed_df = (
    spark.read
    .option("header", True)
    .schema(orders_schema)
    .csv(str(orders_path))
)

orders_typed_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [38]:
orders_typed_df.show(5, truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|delivered   |2017-10-02 10:56:33     |2017-10-02 11:07:15|2017-10-04 19:55:00         |2017-10-10 21:25:13          |2017-10-18 00:00:00          |
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|delivered   |2018-07-24 20:41:37     |2018-07-26 03:24:27|2018-07-26 14:3

In [39]:
print("Rows: ", orders_typed_df.count())
print("Columns: ", len(orders_typed_df.columns))

Rows:  99441
Columns:  8


In [40]:
orders_typed_df.select([
    F.sum(F.col(col_name).isNull().cast("int")).alias(col_name)
    for col_name in orders_typed_df.columns
]).show(truncate=False)

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|0       |0          |0           |0                       |160              |1783                        |2965                         |0                            |
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [41]:
clean_orders_df = (
    orders_typed_df
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
    .withColumn(
        "order_status",
        F.lower(F.trim(F.col("order_status")))
    )
    .dropDuplicates(["order_id"])
)

In [42]:
clean_orders_df.count()

99441

In [43]:
allowed_statuses = [
    "created",
    "approved",
    "invoiced",
    "processing",
    "shipped",
    "delivered",
    "unavailable",
    "canceled"
]

invalid_status_orders = clean_orders_df.filter(
    ~F.col("order_status").isin(allowed_statuses)
)

In [44]:
invalid_status_orders.groupBy("order_status").count().show()

+------------+-----+
|order_status|count|
+------------+-----+
+------------+-----+



In [45]:
clean_orders_df.columns

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

In [46]:
clean_orders_df = (
    clean_orders_df
    .withColumn(
        "delivery_days",
        F.datediff(
            F.col("order_delivered_customer_date"),
            F.col("order_purchase_timestamp")
        )
    )
    .withColumn(
        "delivery_delay_days",
        F.datediff(
            F.col("order_delivered_customer_date"),
            F.col("order_estimated_delivery_date")
        )
    )
    .withColumn(
        "is_late",
        F.when(
            F.col("delivery_delay_days") > 0,
            F.lit(True)
        ).otherwise(F.lit(False))
    )
)

In [47]:
clean_orders_df.show(


)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|delivery_days|delivery_delay_days|is_late|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-------+
|00018f77f2f0320c5...|f6dd3ec061db4e398...|   delivered|     2017-04-26 10:53:06|2017-04-26 11:05:13|         2017-05-04 14:35:00|          2017-05-12 16:04:24|          2017-05-15 00:00:00|           16|                 -3|  false|
|00042b26cf59d7ce6...|58dbd0b2d70206bf4...|   delivered|     2017-02

In [48]:
clean_orders_df = clean_orders_df.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_days",
    "delivery_delay_days",
    "is_late"
)

In [49]:
clean_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- delivery_delay_days: integer (nullable = true)
 |-- is_late: boolean (nullable = false)



In [50]:
clean_orders_df.show(5, truncate=False)


+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|delivery_days|delivery_delay_days|is_late|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-------+
|00018f77f2f0320c557190d7a144bdd3|f6dd3ec061db4e3987629fe6b26e5cce|delivered   |2017-04-26 10:53:06     |2017-04-26 11:05:13|2017-05-04 14:35:00         |2017-05-12 16:04:24          |2017-05-15 00:00:00          |16           |-

In [51]:
print("Clean row count:", clean_orders_df.count())

print(
    "Distinct order IDs:",
    clean_orders_df.select("order_id").distinct().count()
)

Clean row count: 99441
Distinct order IDs: 99441


In [52]:
clean_orders_df.filter(
    F.col("delivery_days") < 0
).show(5, truncate=False)

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|delivery_days|delivery_delay_days|is_late|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-------+
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-------+



In [53]:
clean_orders_path = Path("../data/silver/orders").resolve()
print(clean_orders_path)

/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/data/silver/orders


In [54]:
clean_orders_df.write.mode("overwrite").parquet(str(clean_orders_path))

26/08/03 15:10:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


In [55]:
saved_orders_df = spark.read.parquet(str(clean_orders_path))
saved_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- delivery_delay_days: integer (nullable = true)
 |-- is_late: boolean (nullable = true)



In [56]:
saved_orders_df.show(5, truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|delivery_days|delivery_delay_days|is_late|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-------+
|00048cc3ae777c65dbb7d2a0634bc1ea|816cbea969fe5b689b39cfc97a506742|delivered   |2017-05-15 21:42:34     |2017-05-17 03:55:27|2017-05-17 11:05:55         |2017-05-22 13:44:35          |2017-06-06 00:00:00          |7            |-

In [57]:
customers_path = Path("../data/raw/olist/olist_customers_dataset.csv").resolve()
print(customers_path)

/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/data/raw/olist/olist_customers_dataset.csv


In [58]:
customers_raw_df = spark.read.option("header", True).option("inferSchema", True).csv(str(customers_path))
customers_raw_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [59]:
customers_raw_df.show(10, truncate=False)

+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|14409                   |franca               |SP            |
|18955e83d337fd6b2def6b18a428ac77|290c77bc529b7ac935b93aa66c333dc3|9790                    |sao bernardo do campo|SP            |
|4e7b3e00288586ebd08712fdd0374a03|060e732b5b29e8181a18229c7b0b2b5e|1151                    |sao paulo            |SP            |
|b2b6027bc5c5109e529d4dc6358b12c3|259dac757896d24d7702b9acbbff3f3c|8775                    |mogi das cruzes      |SP            |
|4f2d8ab171c80ec8364f7c12e35b23ad|345ecd01c38d18a9036ed96c73b8d066|13056                  

In [60]:
print("Rows: ", customers_raw_df.count())
print("Columns: ", len(customers_raw_df.columns))
print("Column names: ", customers_raw_df.columns)

Rows:  99441
Columns:  5
Column names:  ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


In [61]:
customer_row_count = customers_raw_df.count()

distinct_customer_id_row_count = (
    customers_raw_df.select("customer_id").distinct().count()
)

print("Rows: ", customer_row_count)
print("Distinct customer IDs: ", distinct_customer_id_row_count)
print(f"customer_id is unique: {customer_row_count == distinct_customer_id_row_count}")

Rows:  99441
Distinct customer IDs:  99441
customer_id is unique: True


In [62]:
distinct_unique_customer_count = (
    customers_raw_df.select("customer_unique_id").distinct().count()
)

print("Rows: ", customer_row_count)
print("Distinct unique customer IDs: ", distinct_unique_customer_count)
print(f"customer_id is unique: {customer_row_count == distinct_unique_customer_count}")

Rows:  99441
Distinct unique customer IDs:  96096
customer_id is unique: False


In [63]:
repeated_customers = (
    customers_raw_df
    .groupBy("customer_unique_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc())
)

repeated_customers.show(10, truncate=False)

+--------------------------------+-----+
|customer_unique_id              |count|
+--------------------------------+-----+
|8d50f5eadf50201ccdcedfb9e2ac8455|17   |
|3e43e6105506432c953e165fb2acf44c|9    |
|6469f99c1f9dfae7733b25662e7f1782|7    |
|1b6c7548a2a1f9037c1fd3ddfed95f33|7    |
|ca77025e7201e3b30c44b472ff346268|7    |
|47c1a3033b8b77b3ab6e109eb4d5fdf3|6    |
|f0e310a6839dce9de1638e0fe5ab282a|6    |
|dc813062e0fc23409cd255f7f53c7074|6    |
|12f5d6e1cbf93dafd9dcc19095df0b3d|6    |
|63cfc61cee11cbe306bff5857d00bfe4|6    |
+--------------------------------+-----+
only showing top 10 rows


In [64]:
customers_schema = StructType([
    StructField("customer_id", StringType(), nullable=False),
    StructField("customer_unique_id", StringType(), nullable=False),
    StructField("customer_zip_code_prefix", StringType(), nullable=True),
    StructField("customer_city", StringType(), nullable=True),
    StructField("customer_state", StringType(), nullable=True)
])

customers_typed_df = spark.read.option("header", True).schema(customers_schema).csv(str(customers_path))

In [65]:
customers_typed_df.show(10, truncate=False)

+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|14409                   |franca               |SP            |
|18955e83d337fd6b2def6b18a428ac77|290c77bc529b7ac935b93aa66c333dc3|09790                   |sao bernardo do campo|SP            |
|4e7b3e00288586ebd08712fdd0374a03|060e732b5b29e8181a18229c7b0b2b5e|01151                   |sao paulo            |SP            |
|b2b6027bc5c5109e529d4dc6358b12c3|259dac757896d24d7702b9acbbff3f3c|08775                   |mogi das cruzes      |SP            |
|4f2d8ab171c80ec8364f7c12e35b23ad|345ecd01c38d18a9036ed96c73b8d066|13056                  

In [66]:
customers_typed_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [67]:
customers_clean_df = (
    customers_typed_df
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("customer_unique_id").isNotNull())
    .withColumn(
        "customer_city",
        F.lower(F.trim(F.col("customer_city")))
    )
    .withColumn(
        "customer_state",
        F.lower(F.trim(F.col("customer_state")))
    )
    .dropDuplicates(["customer_id"])
)

In [68]:
customers_clean_df.show(10, truncate=False)

+--------------------------------+--------------------------------+------------------------+-------------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city      |customer_state|
+--------------------------------+--------------------------------+------------------------+-------------------+--------------+
|00050bf6e01e69d5c0fd612f1bcfb69c|e3cf594a99e810f58af53ed4820f25e5|98700                   |ijui               |rs            |
|000598caf2ef4117407665ac33275130|7e0516b486e92ed3f3afdd6d1276cfbd|35540                   |oliveira           |mg            |
|0013cd8e350a7cc76873441e431dd5ee|334fed5abcee3aa96c13f1432703e1fd|03585                   |sao paulo          |sp            |
|0015bc9fd2d5395446143e8b215d7c75|490c854539b21598cfbbac518ca25788|12233                   |sao jose dos campos|sp            |
|001df1ee5c36767aa607001ab1a13a06|46b44ab325f78e5bb3dc0bbef1082082|01030                   |sao paulo   

In [69]:
customers_clean_df.groupBy("customer_state").count().orderBy(F.col("count").desc()).show(30)

+--------------+-----+
|customer_state|count|
+--------------+-----+
|            sp|41746|
|            rj|12852|
|            mg|11635|
|            rs| 5466|
|            pr| 5045|
|            sc| 3637|
|            ba| 3380|
|            df| 2140|
|            es| 2033|
|            go| 2020|
|            pe| 1652|
|            ce| 1336|
|            pa|  975|
|            mt|  907|
|            ma|  747|
|            ms|  715|
|            pb|  536|
|            pi|  495|
|            rn|  485|
|            al|  413|
|            se|  350|
|            to|  280|
|            ro|  253|
|            am|  148|
|            ac|   81|
|            ap|   68|
|            rr|   46|
+--------------+-----+



In [70]:
invalid_state_codes = customers_clean_df.filter(F.length(F.col("customer_state")) != 2)

invalid_state_codes.show(truncate=False)

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
+-----------+------------------+------------------------+-------------+--------------+



In [71]:
customers_output_path = Path("../data/silver/customers").resolve()
customers_clean_df.coalesce(2).write.mode("overwrite").parquet(str(customers_output_path))

In [72]:
saved_customers_df = spark.read.parquet(str(customers_output_path))
saved_customers_df.show(10, truncate=False)

+--------------------------------+--------------------------------+------------------------+----------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city   |customer_state|
+--------------------------------+--------------------------------+------------------------+----------------+--------------+
|000bf8121c3412d3057d32371c5d3395|1bc9b2dad6aefbfbc011508e34c8adfc|12335                   |jacarei         |sp            |
|00114026c1b7b52ab1773f317ef4880b|f4dc0a81a11d3d270ccf5a9c4b5b187b|22470                   |rio de janeiro  |rj            |
|0015f7887e2fde13ddaa7b8e385af919|866c923cde750dfc8cfbcf9d5ced0ee4|25903                   |mage            |rj            |
|001f6f1a5e902ad14e1f709a7215de11|c6b7dcd3718d1ad87f069d32a8566ce2|12460                   |campos do jordao|sp            |
|002348c1099e3229276c8ad7d4ddc702|934c19eeef04da89928f995df85cf3f8|13295                   |itupeva         |sp            |
